# Environment Setup - E-Commerce Agent Workshop

This notebook sets up all the prerequisites for the E-Commerce Agent Evaluation & Observability Workshop. We'll provision AWS infrastructure and verify that everything is ready for the hands-on exercises.

## What This Workshop Covers

This workshop demonstrates building a production-ready **single Product Catalog Agent with RBAC** (Role-Based Access Control) using:
- **Strands Agent SDK** - Framework for building AI agents
- **Amazon Bedrock** - Managed AI service with Claude models
- **AWS AgentCore** - Production deployment platform
- **DynamoDB** - NoSQL database for product catalog, orders, and accounts

## Infrastructure Overview

The setup script will create:

### 1. DynamoDB Tables
- **ecommerce-workshop-products** - Product catalog with inventory
- **ecommerce-workshop-orders** - Customer orders with status tracking
- **ecommerce-workshop-accounts** - Customer account information

### 2. Sample Data
- Pre-populated products, orders, and accounts for testing
- Realistic e-commerce scenarios

### 3. SSM Parameters
- Resource discovery parameters for agents to find tables
- Configuration management

## Prerequisites

Before running this notebook, ensure you have:
- AWS credentials configured (via AWS CLI or environment variables)
- Bedrock model access enabled for Claude Sonnet 4.6
- Appropriate IAM permissions for DynamoDB, SSM, and Bedrock

---

## Step 1: Python Runtime Setup

Please make sure that you've done the [Quick start](../README.md#quick-start) so as to configure the Python runtime.

## Step 2: Provision AWS Infrastructure

Now we'll run the infrastructure setup script. This will:
1. Create three DynamoDB tables with appropriate indexes
2. Load sample data (orders, accounts, products)
3. Create SSM parameters for resource discovery

The script is idempotent - it's safe to run multiple times.

In [1]:
# Run infrastructure setup
!python setup_infrastructure.py

Infrastructure Setup initialized
  Region: us-west-2
  Account: 534409838809
  Prefix: ecommerce-workshop

Starting Infrastructure Setup

1. Creating DynamoDB Tables...
  ✅ ecommerce-workshop-orders: Already exists
  ✅ ecommerce-workshop-accounts: Already exists

2. Loading Sample Data...
  ✅ Loaded 10 orders
  ✅ Loaded 8 accounts

3. Creating Products Table...
  ✅ ecommerce-workshop-products: Already exists

4. Loading Product Data...
  ✅ Loaded 10 products
  ✅ Loaded store policies

5. Creating SSM Parameters...
  ✅ ecommerce-workshop-orders-table: ecommerce-workshop-orders
  ✅ ecommerce-workshop-accounts-table: ecommerce-workshop-accounts
  ✅ ecommerce-workshop-products-table: ecommerce-workshop-products

INFRASTRUCTURE SETUP COMPLETE!

Resources created:
  - DynamoDB: ecommerce-workshop-orders
  - DynamoDB: ecommerce-workshop-accounts
  - DynamoDB: ecommerce-workshop-products

Run verify_infrastructure.py to confirm setup


## Step 3: Verify Infrastructure Setup

Finally, we'll verify that all resources were created successfully. The verification script checks:
- ✅ AWS identity and permissions
- ✅ DynamoDB tables exist and contain data
- ✅ Bedrock model access (Claude Sonnet 4.6)
- ✅ SSM parameters are configured

If all checks pass, you're ready to proceed with the workshop!

In [2]:
# Verify infrastructure is ready
!python verify_infrastructure.py


E-Commerce Agent Workshop - Infrastructure Verification

AWS Region: us-west-2

1. Checking AWS Identity...
  ✅ AWS Identity: arn:aws:iam::534409838809:user/Admin
  ✅ Account: 534409838809

2. Checking DynamoDB Tables...
  ✅ ecommerce-workshop-orders: ACTIVE (10 items)
  ✅ ecommerce-workshop-accounts: ACTIVE (8 items)
  ✅ ecommerce-workshop-products: ACTIVE (13 items)

3. Checking Bedrock Model Access...
  ✅ Model: anthropic.claude-sonnet-4-6
   Note: Workshop uses global inference profile:
   - global.anthropic.claude-sonnet-4-6

4. Checking SSM Parameters...
  ✅ Parameter: ecommerce-workshop-orders-table
  ✅ Parameter: ecommerce-workshop-accounts-table
  ✅ Parameter: ecommerce-workshop-products-table

✅ All infrastructure checks PASSED!
You are ready to start the workshop.



## Step 4: Explore the Sample Dataset

Before diving into building agents, it's helpful to understand the data they'll be working with. The three DynamoDB tables are pre-populated with realistic e-commerce scenarios designed to cover common customer service situations.

### ecommerce-workshop-accounts
Customer profiles with:
- **Membership tiers**: `standard`, `gold`, `platinum` — agents may need to offer tier-appropriate responses
- **Account status**: `active` or `suspended` — agents should check this before processing requests
- **Payment methods and preferences**: realistic multi-field records

### ecommerce-workshop-orders
10 orders covering every major order lifecycle status:
- `pending`, `processing`, `shipped`, `delivered` — the happy path
- `cancelled`, `return_requested`, `refunded` — escalation and resolution flows
- Each order links to a customer via `customer_id` and references products by `product_id`

### ecommerce-workshop-products
Product catalog across 7 categories (Audio, Wearables, Gaming, Monitors, Cameras, Accessories, Furniture) with:
- Stock levels (one product intentionally out of stock)
- Warranties and per-product return policies
- Store-wide policies for returns, shipping, and price matching

Run the cells below to browse the data as DataFrames.

In [3]:

import json
import textwrap
import pandas as pd
from IPython.display import display

# Load sample data from local files (no AWS connection needed)
with open("sample_data/orders.json") as f:
    orders_data = json.load(f)

with open("sample_data/accounts.json") as f:
    accounts_data = json.load(f)

with open("sample_data/products.json") as f:
    products_data = json.load(f)

print("Sample data loaded successfully!")
print(f"  - {len(accounts_data['accounts'])} customer accounts")
print(f"  - {len(orders_data['orders'])} orders")
print(f"  - {len(products_data['products'])} products in the catalog")


Sample data loaded successfully!
  - 8 customer accounts
  - 10 orders
  - 10 products in the catalog


In [4]:

# Display customer accounts with membership tier breakdown
accounts_df = pd.DataFrame([
    {
        "customer_id": a["customer_id"],
        "name": f"{a['first_name']} {a['last_name']}",
        "membership_tier": a["membership_tier"],
        "account_status": a["account_status"],
        "total_orders": a["total_orders"],
        "total_spent": f"${a['total_spent']:,.2f}",
        "member_since": a["created_date"],
    }
    for a in accounts_data["accounts"]
])

print("=== Customer Accounts ===")
display(accounts_df)

print("\n--- Membership Tier Distribution ---")
tier_counts = accounts_df["membership_tier"].value_counts().rename("count").to_frame()
display(tier_counts)

print("\n--- Account Status Distribution ---")
status_counts = accounts_df["account_status"].value_counts().rename("count").to_frame()
display(status_counts)


=== Customer Accounts ===


,customer_id,name,membership_tier,account_status,total_orders,total_spent,member_since
0,CUST-1001,John Smith,gold,active,15,"$2,456.78",2022-03-15
1,CUST-1002,Sarah Johnson,platinum,active,42,"$8,934.56",2021-08-22
2,CUST-1003,Mike Wilson,standard,active,3,$549.99,2024-01-10
3,CUST-1004,Emily Davis,gold,active,12,"$1,876.43",2023-05-18
4,CUST-1005,Robert Brown,standard,active,5,$623.45,2024-06-01
5,CUST-1006,Lisa Martinez,standard,suspended,8,"$1,245.67",2023-11-20
6,CUST-1007,David Lee,gold,active,23,"$4,567.89",2022-09-30
7,CUST-1008,Jennifer Taylor,standard,active,2,$399.98,2024-10-05



--- Membership Tier Distribution ---


,count
membership_tier,
standard,4
gold,3
platinum,1



--- Account Status Distribution ---


,count
account_status,
active,7
suspended,1


In [5]:

# Display orders with status breakdown
orders_df = pd.DataFrame([
    {
        "order_id": o["order_id"],
        "customer_id": o["customer_id"],
        "status": o["status"],
        "order_date": o["order_date"],
        "items": len(o["items"]),
        "total": f"${o['total']:,.2f}",
    }
    for o in orders_data["orders"]
])

print("=== Orders ===")
display(orders_df)

print("\n--- Order Status Distribution ---")
print("(These represent the different customer service scenarios agents will handle)")
status_counts = orders_df["status"].value_counts().rename("count").to_frame()
display(status_counts)

print("\n--- Orders per Customer ---")
orders_per_customer = (
    orders_df.groupby("customer_id")
    .agg(num_orders=("order_id", "count"))
    .sort_values("num_orders", ascending=False)
)
display(orders_per_customer)


=== Orders ===


,order_id,customer_id,status,order_date,items,total
0,ORD-2024-10001,CUST-1001,delivered,2024-12-15,1,$79.99
1,ORD-2024-10002,CUST-1002,shipped,2024-12-28,2,$359.97
2,ORD-2024-10003,CUST-1003,processing,2025-01-02,1,$449.99
3,ORD-2024-10004,CUST-1001,delivered,2024-12-01,2,$89.98
4,ORD-2024-10005,CUST-1004,pending,2025-01-05,1,$599.99
5,ORD-2024-10006,CUST-1005,return_requested,2024-12-20,1,$149.99
6,ORD-2024-10007,CUST-1006,refunded,2024-11-25,1,$199.99
7,ORD-2024-10008,CUST-1002,delivered,2024-10-15,2,$129.98
8,ORD-2024-10009,CUST-1007,shipped,2025-01-06,2,$189.98
9,ORD-2024-10010,CUST-1008,cancelled,2024-12-30,1,$199.99



--- Order Status Distribution ---
(These represent the different customer service scenarios agents will handle)


,count
status,
delivered,3
shipped,2
processing,1
pending,1
return_requested,1
refunded,1
cancelled,1



--- Orders per Customer ---


,num_orders
customer_id,
CUST-1001,2
CUST-1002,2
CUST-1003,1
CUST-1004,1
CUST-1005,1
CUST-1006,1
CUST-1007,1
CUST-1008,1


In [6]:

# Display product catalog and store policies
products_df = pd.DataFrame([
    {
        "product_id": p["product_id"],
        "name": p["name"],
        "category": p["category"],
        "price": f"${p['price']:.2f}",
        "in_stock": "Yes" if p["in_stock"] else "No",
        "stock_qty": p["stock_quantity"],
        "rating": p["rating"],
        "warranty": p["warranty"],
    }
    for p in products_data["products"]
])

print("=== Product Catalog ===")
display(products_df)

print("\n--- Products by Category ---")
category_summary = (
    products_df.assign(price_raw=[p["price"] for p in products_data["products"]])
    .groupby("category")
    .agg(count=("product_id", "count"), avg_price=("price_raw", "mean"))
    .rename(columns={"count": "# Products", "avg_price": "Avg Price ($)"})
    .round(2)
)
display(category_summary)

print("\n=== Store Policies ===")
for key, policy_text in products_data["policies"].items():
    print(f"\n{key.replace('_', ' ').title()}:")
    # Word-wrap at ~100 chars
    import textwrap
    for line in textwrap.wrap(policy_text, width=100):
        print(f"  {line}")


=== Product Catalog ===


,product_id,name,category,price,in_stock,stock_qty,rating,warranty
0,PROD-001,Wireless Bluetooth Headphones,Audio,$79.99,Yes,150,4.5,1 year manufacturer warranty
1,PROD-015,Smart Watch Pro,Wearables,$299.99,Yes,75,4.7,2 year manufacturer warranty
2,PROD-016,Watch Band - Leather,Accessories,$29.99,Yes,200,4.3,90 days
3,PROD-042,"4K Ultra HD Monitor 27""",Monitors,$449.99,Yes,45,4.6,3 year manufacturer warranty
4,PROD-008,USB-C Hub 7-in-1,Accessories,$49.99,Yes,300,4.4,1 year
5,PROD-055,Noise Canceling Earbuds,Audio,$149.99,Yes,120,4.5,1 year
6,PROD-101,Ergonomic Office Chair,Furniture,$599.99,Yes,25,4.8,"5 year warranty on frame, 2 years on components"
7,PROD-077,Gaming Mechanical Keyboard RGB,Gaming,$159.99,Yes,60,4.6,2 years
8,PROD-088,Webcam 4K HDR,Cameras,$199.99,No,0,4.7,2 years
9,PROD-033,Portable Bluetooth Speaker,Audio,$199.99,Yes,80,4.4,1 year



--- Products by Category ---


,# Products,Avg Price ($)
category,,
Accessories,2,39.99
Audio,3,143.32
Cameras,1,199.99
Furniture,1,599.99
Gaming,1,159.99
Monitors,1,449.99
Wearables,1,299.99



=== Store Policies ===

General Return Policy:
  Most items can be returned within 30 days of delivery for a full refund. Items must be in original
  condition with all packaging and accessories. Some categories (hygiene products, opened software)
  may have restrictions.

Warranty Claims:
  To file a warranty claim, contact customer service with your order number and description of the
  issue. We may request photos or videos of the defect. Approved claims will receive repair,
  replacement, or refund at our discretion.

Price Match:
  We offer price matching within 14 days of purchase if you find a lower price from an authorized
  retailer. Contact customer service with proof of the lower price.

Shipping Policy:
  Standard shipping (5-7 business days) is free for orders over $50. Express shipping (2-3 business
  days) is $9.99. Overnight shipping is $19.99. Alaska and Hawaii may have additional fees.


---

## Next Steps

If all verification checks passed, you're ready to start the workshop!

Proceed to:
- **01-single-agent-prototype** - Build a Product Catalog Agent with RBAC
- **02-evaluation-baseline** - Establish evaluation metrics and baselines
- **03-production-deployment** - Deploy to AWS AgentCore
- **04-online-eval-observability** - Monitor and evaluate production agents
- **05-production-batch-evaluation** - Batch evaluation, drift detection, and feedback loops

## Troubleshooting

If verification fails:

### DynamoDB Tables Not Found
```bash
# Re-run setup
python setup_infrastructure.py
```

### Bedrock Model Access Issues
1. Go to AWS Console → Bedrock → Model Access
2. Request access to Claude Sonnet 4.6
3. Wait for approval (usually instant)

### AWS Credentials Not Configured
```bash
# Configure AWS CLI
aws configure
```

### Clean Up Resources
After completing the workshop, remove all infrastructure:
```bash
python setup_infrastructure.py --cleanup
```

---

**Ready to build intelligent agents? Let's go!**